# Plotting Experiment 2

In [ ]:
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [ ]:
def get_dz(ds):
    dz = -1 * ds.z_ifc.diff('height_2')
    dz = dz.rename({'height_2':'height'})
    dz = dz.assign_coords(height=(dz.height - 1))
    return dz

def tracer_load(tr,rho,dz):
    m_tr = tr * rho * dz
    m_tr = m_tr.sum('height')
    m_tr = m_tr.squeeze()
    return m_tr

## Load the model data

In [ ]:
ds1 = xr.open_mfdataset('/path/to/noARI/icon-art-dust-remap_DOM01_ML_*', autoclose=True)
ds2 = xr.open_mfdataset('/path/to/ARI/icon-art-dust-remap_DOM01_ML_*', autoclose=True)
dz  = get_dz(ds1)

dustload1 = tracer_load(ds1.dusta + ds1.dustb + ds1.dustc, ds1.rho, dz)
dustload2 = tracer_load(ds2.dusta + ds2.dustb + ds2.dustc, ds2.rho, dz)

E_no_ari  = (ds1.emiss_dusta + ds1.emiss_dustb + ds1.emiss_dustc)
E_ari     = (ds2.emiss_dusta + ds2.emiss_dustb + ds2.emiss_dustc)

Temp      = (ds2.temp - ds1.temp) # ARI - noARI
delta_T   = Temp[:,89,:,:]

In [ ]:
dustload1 = dustload1.where(dustload1>0.1E6)
dustload2 = dustload2.where(dustload2>0.1E6)
E_no_ari  = E_no_ari.where(E_no_ari>0.0)
E_ari     = E_ari.where(E_ari>0.0)

## Prepare Map

In [ ]:
lat = ds1.lat
lon = ds1.lon
x,y = np.meshgrid(lon,lat)
parallels = np.arange(-90,90,30)
meridians = np.arange(-180,180,90)
crs_proj = ccrs.PlateCarree()

## Plot: Dust emissions

In [ ]:
# Part 1
fig1 = plt.figure(figsize=(9,9))
ax1 = plt.subplot(3,2,1,projection=crs_proj)
cont = plt.contourf(x,y,dustload1[12,:,:]*1E-6,cmap='plasma_r',levels=np.arange(0,3,0.25),transform=ccrs.PlateCarree(),extend='both')
ax1.coastlines(resolution='50m')
plt.colorbar(cont)
plt.title('Dust after 12 hours (g/m2)')

ax2 = plt.subplot(3,2,2,projection=crs_proj)
cont = plt.contourf(x,y,dustload1[24,:,:]*1E-6,cmap='plasma_r',levels=np.arange(0,3,0.25),transform=ccrs.PlateCarree(),extend='both')
ax2.coastlines(resolution='50m')
plt.colorbar(cont)
plt.title('Dust after 24 hours (g/m2)')

ax3 = plt.subplot(3,2,3,projection=crs_proj)
cont = plt.contourf(x,y,E_no_ari[12,:,:],cmap='plasma_r',levels=np.arange(0,50,5)*1E-6,transform=ccrs.PlateCarree(),extend='both')
ax3.coastlines(resolution='50m')
plt.colorbar(cont)
plt.title('E after 12 hours (kg/(m2 s))')

ax4 = plt.subplot(3,2,4,projection=crs_proj)
cont = plt.contourf(x,y,E_no_ari[24,:,:],cmap='plasma_r',levels=np.arange(0,50,5)*1E-6,transform=ccrs.PlateCarree(),extend='both')
ax4.coastlines(resolution='50m')
plt.colorbar(cont)
plt.title('E after 24 hours (kg/(m2 s))')

plt.subplot(3,2,(5,6))
plt.plot(np.arange(0,25),(E_no_ari.mean(axis=1,skipna=True)).mean(axis=1,skipna=True),color='k',label='Global')
plt.plot(np.arange(0,25),(E_no_ari[:,100:181,320:501].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='N. Africa')
plt.plot(np.arange(0,25),(E_no_ari[:,200:261,580:681].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='Australia')
plt.plot(np.arange(0,25),(E_no_ari[:,100:280,120:240].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='America')
plt.legend(loc='upper left')
plt.xlabel('time (UTC)')
plt.title('Mean dust emission in kg/(m2 s)')
plt.grid()
plt.xlim(0,24)

plt.show()
#fig1.savefig('plot_ex2_1.png',dpi=300)

## Plots: Differences with and without Aerosol-Radiation interaction

In [ ]:
# Part 2.1
fig2 =plt.figure(figsize=(10,5))
ax1 = plt.subplot(2,2,1,projection=crs_proj)
cont = plt.contourf(x,y,delta_T[12,:,:],cmap='seismic',levels=np.arange(-2,2.1,0.25),transform=ccrs.PlateCarree(),extend='both')
ax1.coastlines(resolution='50m')
plt.colorbar(cont)
plt.title('$\Delta$T after 12 hours (K)')

ax2 = plt.subplot(2,2,2,projection=crs_proj)
cont = plt.contourf(x,y,delta_T[24,:,:],cmap='seismic',levels=np.arange(-2,2.1,0.25),transform=ccrs.PlateCarree(),extend='both')
ax2.coastlines(resolution='50m')
plt.colorbar(cont)
plt.title('$\Delta$T after 24 hours (K)')

ax3 = plt.subplot(2,2,3,projection=crs_proj)
cont = plt.contourf(x,y,(E_ari[12,:,:]-E_no_ari[12,:,:]),cmap='seismic',levels=np.arange(-3,3.1,0.25)*1E-6,transform=ccrs.PlateCarree(),extend='both')
ax3.coastlines(resolution='50m')
plt.colorbar(cont)
plt.title('$\Delta$E after 12 hours (kg/(m2 s))')

ax4 = plt.subplot(2,2,4,projection=crs_proj)
cont = plt.contourf(x,y,(E_ari[24,:,:]-E_no_ari[24,:,:]),cmap='seismic',levels=np.arange(-3,3.1,0.25)*1E-6,transform=ccrs.PlateCarree(),extend='both')
ax4.coastlines(resolution='50m')
plt.colorbar(cont)
plt.title('$\Delta$E after 24 hours (kg/(m2 s))')

plt.show()
#fig2.savefig('plot_ex2_2.png',dpi=300)

In [ ]:
# Part 2.2
fig3,ax3=plt.subplots(nrows=2, ncols=1, figsize=(9,6))
plt.subplot(2,1,1)
plt.plot(np.arange(0,25),((E_ari-E_no_ari).mean(axis=1,skipna=True)).mean(axis=1,skipna=True),color='k',label='Global')
plt.plot(np.arange(0,25),((E_ari-E_no_ari)[:,100:181,320:501].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='N. Africa')
plt.plot(np.arange(0,25),((E_ari-E_no_ari)[:,200:261,580:681].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='Australia')
plt.plot(np.arange(0,25),((E_ari-E_no_ari)[:,100:280,120:240].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='America')
plt.grid()
plt.xlim(0,24)
#plt.ylim(-50,50)
plt.title('Mean $\Delta$E in kg/(m2 s)')

plt.subplot(2,1,2)
plt.plot(np.arange(0,25),((delta_T).mean(axis=1,skipna=True)).mean(axis=1,skipna=True),color='k',label='Global')
plt.plot(np.arange(0,25),(delta_T[:,100:181,320:501].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='N. Africa')
plt.plot(np.arange(0,25),(delta_T[:,200:261,580:681].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='Australia')
plt.plot(np.arange(0,25),(delta_T[:,100:280,120:240].mean(axis=1,skipna=True)).mean(axis=1,skipna=True),label='America')

plt.grid()
plt.xlim(0,24)
plt.ylim(-0.25,0.25)
plt.title('$\Delta$T in K')

plt.show()
#fig3.savefig('plot_ex2_3.png',dpi=300)